# Анализ сообщений из Телеграмм-канала 

Скрипт для выгрузки сообщений канала был взят из статьи [Пишем простой граббер для Telegram чатов на Python](https://proglib.io/p/pishem-prostoy-grabber-dlya-telegram-chatov-na-python-2019-11-06).

С его помощью выгружены все сообщения канала [MarketTwits](https://t.me/markettwits) в файл channel_messages.json.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
import mplfinance as fplt
import pandas_ta as ta

## Загрузка и первичный обзор данных

In [2]:
df_msgs = pd.read_json('./channel_messages.json')
df_msgs.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 315305 entries, 0 to 315304
Data columns (total 44 columns):
 #   Column                      Non-Null Count   Dtype              
---  ------                      --------------   -----              
 0   _                           315305 non-null  object             
 1   id                          315305 non-null  int64              
 2   peer_id                     315305 non-null  object             
 3   date                        315305 non-null  datetime64[ns, UTC]
 4   message                     314601 non-null  object             
 5   out                         315305 non-null  bool               
 6   mentioned                   315305 non-null  bool               
 7   media_unread                315305 non-null  bool               
 8   silent                      315305 non-null  bool               
 9   post                        315305 non-null  bool               
 10  from_scheduled              314601 non-null 

In [3]:
df_msgs.head(10)

,_,id,peer_id,date,message,out,mentioned,media_unread,silent,post,...,reactions,restriction_reason,ttl_period,quick_reply_shortcut_id,effect,factcheck,report_delivery_until_date,paid_message_stars,action,reactions_are_possible
0,Message,322982,"{'_': 'PeerChannel', 'channel_id': 1203560567}",2025-05-02 16:30:25+00:00,⚠️🇺🇸#недвижимость #сша \nЦены на жилье во Флор...,False,False,False,False,True,...,NaN,[],NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Message,322981,"{'_': 'PeerChannel', 'channel_id': 1203560567}",2025-05-02 16:21:20+00:00,🚫✴️#делистинг #крипто \n16 мая Coinbase остано...,False,False,False,False,True,...,NaN,[],NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Message,322980,"{'_': 'PeerChannel', 'channel_id': 1203560567}",2025-05-02 16:18:39+00:00,"✴️#BTC #satoshinakamoto #крипто \nВ Форнелли, ...",False,False,False,False,True,...,NaN,[],NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Message,322979,"{'_': 'PeerChannel', 'channel_id': 1203560567}",2025-05-02 16:04:50+00:00,💥🇨🇳🇺🇸#пошлины #китай #сша \nКИТАЙ РАССМАТРИВАЕ...,False,False,False,False,True,...,NaN,[],NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Message,322978,"{'_': 'PeerChannel', 'channel_id': 1203560567}",2025-05-02 15:54:28+00:00,"💥🌎#акции #мир \nМировые рынки акций растут, на...",False,False,False,False,True,...,NaN,[],NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,Message,322977,"{'_': 'PeerChannel', 'channel_id': 1203560567}",2025-05-02 15:45:38+00:00,💥🇺🇸#акции #сша \nАмериканские акции отыграли в...,False,False,False,False,True,...,NaN,[],NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,Message,322976,"{'_': 'PeerChannel', 'channel_id': 1203560567}",2025-05-02 15:37:57+00:00,⚠️🌎#экономика #мир #warning \nАктивность в мир...,False,False,False,False,True,...,NaN,[],NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,Message,322975,"{'_': 'PeerChannel', 'channel_id': 1203560567}",2025-05-02 15:18:58+00:00,"✴️#tether #крипто \nCEO Tether сообщил, что на...",False,False,False,False,True,...,NaN,[],NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,Message,322974,"{'_': 'PeerChannel', 'channel_id': 1203560567}",2025-05-02 15:14:20+00:00,⚠️🇷🇺#акции #россия\n\nИНДЕКС МОСБИРЖИ (IMOEX) ...,False,False,False,False,True,...,NaN,[],NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,Message,322973,"{'_': 'PeerChannel', 'channel_id': 1203560567}",2025-05-02 15:10:41+00:00,⚠️🇪🇺#нато #геополитика #впк #европа \nГенсек Н...,False,False,False,False,True,...,NaN,[],NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [4]:
df_msgs['peer_id'].value_counts()

peer_id
{'_': 'PeerChannel', 'channel_id': 1203560567}    315305
Name: count, dtype: int64

In [5]:
df_msgs[df_msgs['_'] == 'MessageService'].head(10)

,_,id,peer_id,date,message,out,mentioned,media_unread,silent,post,...,reactions,restriction_reason,ttl_period,quick_reply_shortcut_id,effect,factcheck,report_delivery_until_date,paid_message_stars,action,reactions_are_possible
205861,MessageService,112205,"{'_': 'PeerChannel', 'channel_id': 1203560567}",2020-12-03 06:36:11+00:00,NaN,False,False,False,True,True,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,{'_': 'MessageActionPinMessage'},1.0
206097,MessageService,111967,"{'_': 'PeerChannel', 'channel_id': 1203560567}",2020-12-02 06:37:52+00:00,NaN,False,False,False,False,True,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,{'_': 'MessageActionPinMessage'},1.0
206368,MessageService,111690,"{'_': 'PeerChannel', 'channel_id': 1203560567}",2020-12-01 06:35:04+00:00,NaN,False,False,False,True,True,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,{'_': 'MessageActionPinMessage'},1.0
206822,MessageService,111231,"{'_': 'PeerChannel', 'channel_id': 1203560567}",2020-11-27 06:35:05+00:00,NaN,False,False,False,True,True,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,{'_': 'MessageActionPinMessage'},1.0
207025,MessageService,111022,"{'_': 'PeerChannel', 'channel_id': 1203560567}",2020-11-26 06:35:21+00:00,NaN,False,False,False,True,True,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,{'_': 'MessageActionPinMessage'},1.0
207232,MessageService,110811,"{'_': 'PeerChannel', 'channel_id': 1203560567}",2020-11-25 06:31:35+00:00,NaN,False,False,False,False,True,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,{'_': 'MessageActionPinMessage'},1.0
207447,MessageService,110593,"{'_': 'PeerChannel', 'channel_id': 1203560567}",2020-11-24 06:34:49+00:00,NaN,False,False,False,True,True,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,{'_': 'MessageActionPinMessage'},1.0
207679,MessageService,110354,"{'_': 'PeerChannel', 'channel_id': 1203560567}",2020-11-23 06:45:22+00:00,NaN,False,False,False,True,True,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,{'_': 'MessageActionPinMessage'},1.0
207943,MessageService,110088,"{'_': 'PeerChannel', 'channel_id': 1203560567}",2020-11-20 06:32:54+00:00,NaN,False,False,False,True,True,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,{'_': 'MessageActionPinMessage'},1.0
208151,MessageService,109872,"{'_': 'PeerChannel', 'channel_id': 1203560567}",2020-11-19 06:40:08+00:00,NaN,False,False,False,True,True,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,{'_': 'MessageActionPinMessage'},1.0


In [6]:
df_msgs['date'].min()

Timestamp('2017-11-09 08:38:22+0000', tz='UTC')